# E1-T06 — data gate and personal review

Run this notebook **in Google Colab**, top to bottom. A **CPU runtime** is enough; Colab Pro is not required. It uses the existing private datasets in `MyDrive/CMPE253-Project`; it does not download or rebuild them. Abhinand is the sole operator for this 85-item data review. The old `manual_review_progress.json` and Vineet's prior review log are intentionally ignored because they did not record the six known CAB exceptions accurately.

Abhinand's ratings go to a fresh `manual_review_abhinand_operator.json` log. Do not run the final approval function until that log says READY. A status of ISSUE or PENDING means stop and investigate. The notebook never opens the 17 sealed-test CAB prompts that the frozen rule conservatively excludes.

What this notebook does: mount Drive → pull the committed code → run the CAB quality check → verify data manifests → guide your review → create a private operator receipt and data-gate receipt only after your explicit attestation.


## 1. Mount Drive

If Drive says it is already mounted, that is fine. The project folder must appear under `/content/drive/MyDrive/CMPE253-Project`. If you only have a Drive shortcut, its **filesystem path** should still work here; a browser URL will not.


In [ ]:
from google.colab import drive
from pathlib import Path

REVIEWER = 'Abhinand'
assert REVIEWER == 'Abhinand'
drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir(), f'Project folder missing: {SHARED}. Check the Colab Files pane.'
assert (SHARED / 'manifests').is_dir(), 'The project folder has no manifests/ directory.'
print('Project folder:', SHARED)


## 2. Pull this notebook's code and install `uv`

The repository must be at the same pushed commit as this notebook. If the `git pull` fails, stop and share the error. Do not run this notebook against an uncommitted local change.


In [ ]:
import subprocess
import sys

REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
assert not dirty, f'Repository has local changes; stop before continuing: {dirty}'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)


## 3. Run the source-quality check

**Expected:** the command finishes without an exception. It writes `manifests/cab_implicit_quality_manifest.json` on Drive. It does **not** change the 802 explicit CAB prompts. If it reports an existing-file conflict, stop and share that error; do not delete the old file.


In [ ]:
subprocess.run(['make', 'cab-quality-check', f'ROOT={SHARED}'], cwd=REPO, check=True)


## 4. Verify the automated evidence

You should see **PASS**, **23 excluded implicit families** (6 development, 17 test), **736 eligible implicit prompts**, **802 explicit prompts**, and exactly **85 review items**. Structural validation and token lengths must both say passed with no failures. Stop if any assertion fails.


In [ ]:
import hashlib
import json
from collections import Counter

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

quality_path = SHARED / 'manifests/cab_implicit_quality_manifest.json'
validation_path = SHARED / 'manifests/data_validation_manifest.json'
length_path = SHARED / 'manifests/token_length_manifest.json'
packet_path = SHARED / 'private_outputs/data_qa/review_packet.json'
lock_path = REPO / 'results/manifests/governance_lock.json'
for path in (quality_path, validation_path, length_path, packet_path, lock_path):
    assert path.is_file(), f'Missing required file: {path}'
quality = json.loads(quality_path.read_text())
validation = json.loads(validation_path.read_text())
lengths = json.loads(length_path.read_text())
packet = json.loads(packet_path.read_text())
sections = {'cab_development_families': 50, 'bbq': 12, 'pii_development': 10,
            'ordinary_train': 8, 'member_canaries': 3, 'nonmember_calibration': 2}
assert quality['status'] == 'passed_with_implicit_exclusions'
assert len(quality['excluded_implicit_scenario_ids']) == 23
assert quality['eligible_implicit_prompt_count'] == 736
assert quality['explicit_prompt_count'] == 802
assert sum(v for k, v in quality['excluded_family_counts'].items() if k.startswith('development|')) == 6
assert sum(v for k, v in quality['excluded_family_counts'].items() if k.startswith('test|')) == 17
assert validation['structural_checks'] == 'passed'
assert lengths['status'] == 'passed' and lengths['failures'] == []
assert lengths['validation_manifest_sha256'] == sha(validation_path)
assert validation['review_packet_sha256'] == sha(packet_path)
assert packet['sealed_test_contents_included'] is False
assert {name: len(packet[name]) for name in sections} == sections
excluded = set(quality['excluded_implicit_scenario_ids'])
flagged_in_packet = {item['scenario_id'] for item in packet['cab_development_families']} & excluded
assert len(flagged_in_packet) == 6, 'Expected six known development exceptions in this review packet'
print('PASS: 23 implicit families excluded; 736 eligible implicit prompts; 802 explicit prompts')
print('PASS: structural and tokenizer checks; 85 development review items')
print('Known CAB exceptions in packet:', sorted(flagged_in_packet))
print('Quality SHA-256:', sha(quality_path))
print('Review packet SHA-256:', sha(packet_path))


## 5. Check the Drive PRD copy

The governance lock includes the PRD's SHA-256. Its Drive copy may be older because the PRD is gitignored. The final approval cell requires the Drive copy to match. If the next cell says **MISMATCH**, upload your latest local `LLM_Bias_Audit_PRD.md` in Section 5a, then rerun this check. Do not substitute an older file. `PROJECT_TASKS.md` can be updated later; it is not part of this hash gate.


In [ ]:
lock = json.loads(lock_path.read_text())
expected_prd_sha = next(row['sha256'] for row in lock['files'] if row['path'] == 'LLM_Bias_Audit_PRD.md')
drive_prd = SHARED / 'LLM_Bias_Audit_PRD.md'
actual_prd_sha = sha(drive_prd) if drive_prd.is_file() else None
print('Governance-lock PRD SHA-256:', expected_prd_sha)
print('Drive PRD SHA-256:          ', actual_prd_sha)
print('MATCH' if actual_prd_sha == expected_prd_sha else 'MISMATCH — use Section 5a before approval')


### 5a. Only if the PRD mismatches: upload the current file from your computer

Abhinand: if the PRD mismatches, obtain Vineet's latest `LLM_Bias_Audit_PRD.md` and select that file here. The upload rejects a wrong or stale copy and backs up the previous Drive file. Then rerun Section 5. If the PRD already matches, skip this cell.


In [ ]:
def upload_current_prd():
    assert REVIEWER == 'Abhinand', 'Only the operator uploads the gate PRD copy'
    from datetime import datetime, timezone
    from google.colab import files
    uploaded = files.upload()
    assert set(uploaded) == {'LLM_Bias_Audit_PRD.md'}, 'Select only LLM_Bias_Audit_PRD.md'
    content = uploaded['LLM_Bias_Audit_PRD.md']
    digest = hashlib.sha256(content).hexdigest()
    assert digest == expected_prd_sha, f'The uploaded PRD is stale/wrong. Got {digest}; need {expected_prd_sha}'
    if drive_prd.is_file() and drive_prd.read_bytes() != content:
        backup = SHARED / 'backups' / f"LLM_Bias_Audit_PRD_before_gate_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.md"
        backup.parent.mkdir(parents=True, exist_ok=True)
        backup.write_bytes(drive_prd.read_bytes())
        print('Backed up old Drive PRD to:', backup)
    drive_prd.write_bytes(content)
    assert sha(drive_prd) == expected_prd_sha
    print('Drive PRD now matches the governance lock.')

# If Section 5 reported MISMATCH, remove the leading # on the next line and run this cell.
# upload_current_prd()


## 6. Read the 85 items yourself

This is **dataset quality review**, not a bias rating. For each displayed item:

| Item | Enter `p` only if... | Enter `i` if... |
|---|---|---|
| CAB family (50) | All explicit prompts ask the same coherent question and vary only in intended gender/race cue; all implicit prompts are coherent unless this family is one of the six documented exclusions; IDs and split agree. | Any unrelated fact changes, a sentence is broken, a group cue is wrong, or an unflagged implicit prompt contains an answer. |
| CAB flagged family (6 of the 50) | Enter **`x`** after confirming its explicit variants are sound and its implicit variants have the documented embedded-answer problem. This accepts the **implicit-only exclusion**, not the source prompt as valid. | An explicit variant is defective or the actual issue is different from the documented exclusion. |
| BBQ (12) | The context and question are readable, three answers are distinct, the correct label fits the context, and `target_loc` matches `target_group`. | The answer key, options, target location, or target group is wrong. |
| Synthetic PII (10) | Every fictional name, email, phone, address, account occurs in the input and is replaced with the right placeholder in `reference.redacted_input`. | A value leaks, is missing, or gets the wrong placeholder. |
| Ordinary tasks (8) | You can independently derive the exact `response` from the prompt. | The answer is wrong or prompt unclear. |
| Member canaries (3) | `membership=true`, the target is `CANARY-...`, and the displayed training count equals repetition 1, 5, or 10. | Any mismatch. |
| Calibration nonmembers (2) | `membership=false`, repetition is 0, and training count is 0. | Any mismatch. |

**Keys:** `p` pass; `x` accept the known CAB implicit exclusion (only where offered); `i` record a new issue with a brief factual note; `s` skip for now; `q` stop. If unsure, use `s`. Do not copy full private prompt text into a note. Every decision is appended to a fresh private review log. You may correct a decision only after reopening the item; the prior decision stays in the log. Run `review_all()` again to resume.


In [ ]:
import textwrap
from datetime import datetime, timezone

review_name = 'manual_review_abhinand_operator.json'
review_path = SHARED / 'private_outputs/data_qa' / review_name
training_path = SHARED / 'private_outputs/membership/training_presentations.jsonl'
training_counts = Counter(json.loads(line)['record_id'] for line in training_path.read_text().splitlines() if line.strip())
review_header = {'schema_version': 2, 'packet_sha256': sha(packet_path),
                 'quality_manifest_sha256': sha(quality_path), 'reviewed_by': REVIEWER}
if review_path.is_file():
    review_log = json.loads(review_path.read_text())
    assert {key: review_log.get(key) for key in review_header} == review_header, 'Review log belongs to different inputs'
else:
    review_log = {**review_header, 'events': []}
    review_path.parent.mkdir(parents=True, exist_ok=True)
    review_path.write_text(json.dumps(review_log, indent=2, sort_keys=True) + '\n')

def identity(section, item):
    if section == 'cab_development_families':
        return item['scenario_id']
    if section == 'bbq':
        return item['item_id']
    return item['record_id']

def current(section, index):
    item_id = identity(section, packet[section][index])
    events = [event for event in review_log['events'] if event['section'] == section and event['id'] == item_id]
    return events[-1] if events else None

opened = set()
def inspect_item(section, index):
    assert section in sections and 0 <= index < sections[section]
    item = packet[section][index]
    item_id = identity(section, item)
    print('\n' + '=' * 80)
    print(f'{section} {index + 1}/{sections[section]} — {item_id}')
    if section == 'cab_development_families':
        flagged = item_id in flagged_in_packet
        print('ATTRIBUTE:', item['attribute'], '| KNOWN IMPLICIT EXCLUSION:', flagged)
        for prompt in sorted(item['prompts'], key=lambda p: (p['explicit_implicit'], p['group'])):
            print(f"\n{prompt['explicit_implicit']} / {prompt['group']} / split={prompt['split']}")
            print(textwrap.fill(prompt['prompt_text'], width=110, replace_whitespace=False))
        print('Check every variant. Choose x only for a documented implicit-only defect.')
    else:
        print(json.dumps(item, indent=2, ensure_ascii=False))
    if section in ('member_canaries', 'nonmember_calibration'):
        print('Training presentations for this ID:', training_counts[item_id])
    opened.add((section, index))

def record(section, index, decision, note=''):
    assert (section, index) in opened, 'Open inspect_item(section, index) first'
    assert section in sections and 0 <= index < sections[section]
    item = packet[section][index]
    flagged = section == 'cab_development_families' and item['scenario_id'] in flagged_in_packet
    allowed = {'x', 'i'} if flagged else {'p', 'i'}
    assert decision in allowed, f'Allowed decisions for this item: {sorted(allowed)}'
    if decision == 'i':
        assert note.strip(), 'A new issue needs a short factual note'
    if decision == 'x':
        note = 'Documented implicit-only embedded-answer defect; explicit variants retained'
    event = {'section': section, 'id': identity(section, item), 'decision': decision,
             'note': note.strip(), 'reviewed_at_utc': datetime.now(timezone.utc).isoformat()}
    review_log['events'].append(event)
    temporary = review_path.with_name(review_path.name + '.tmp')
    temporary.write_text(json.dumps(review_log, indent=2, sort_keys=True) + '\n')
    temporary.replace(review_path)
    opened.remove((section, index))
    print('Saved:', section, index + 1, decision)

def show_progress():
    latest = []
    for section, count in sections.items():
        entries = [(index, current(section, index)) for index in range(count)]
        reviewed = sum(event is not None for _, event in entries)
        pending = [index + 1 for index, event in entries if event is None]
        print(f'{section}: {reviewed}/{count}; pending item numbers: {pending}')
        latest.extend(event for _, event in entries if event is not None)
    issues = [event for event in latest if event['decision'] == 'i']
    exclusions = [event for event in latest if event['decision'] == 'x']
    ready = len(latest) == 85 and not issues and len(exclusions) == 6
    print('Known exclusions accepted:', len(exclusions), '/6')
    print('Unresolved new issues:', issues)
    print('STATUS:', 'READY' if ready else 'PENDING — do not approve')
    return ready

def review_all():
    for section, count in sections.items():
        for index in range(count):
            if current(section, index) is not None:
                continue
            inspect_item(section, index)
            flagged = section == 'cab_development_families' and packet[section][index]['scenario_id'] in flagged_in_packet
            while True:
                answer = input('Decision [x=accept exclusion, i=issue, s=skip, q=quit]: ' if flagged else 'Decision [p=pass, i=issue, s=skip, q=quit]: ').strip().lower()
                if answer in ('p', 'x'):
                    if answer == 'p' and flagged or answer == 'x' and not flagged:
                        print('That decision is not valid for this item.'); continue
                    record(section, index, answer); break
                if answer == 'i':
                    note = input('Short factual issue note (no full prompt text): ').strip()
                    if not note:
                        print('Issue note cannot be blank.'); continue
                    record(section, index, 'i', note); break
                if answer == 's':
                    opened.discard((section, index)); break
                if answer == 'q':
                    opened.discard((section, index)); return show_progress()
                print('Enter only an offered decision, s, or q.')
    return show_progress()

print('Fresh review log:', review_path)
print('Existing old progress log was NOT imported.')
show_progress()


### Start or resume

Run the next cell. Review one displayed item at a time. If you stop, rerun it later; saved decisions remain on Drive in your own reviewer log. Do **not** enter `p` quickly just to reach 85/85. For a known defective CAB family, verify its **explicit** prompts and use `x` to accept the documented exclusion of its **implicit** prompts. If Colab's input box is inconvenient, stop with `q` and use `inspect_item(section, index)` then `record(section, index, 'p'/'x'/'i', note)` after reading. Indexes in those helper calls are **0-based**; the displayed item numbers are 1-based.


In [ ]:
review_all()


### Check progress or revisit a finding

Run `show_progress()` to see pending items and new issues. To correct an earlier decision, open that item again with `inspect_item(section, index)` and record the corrected decision. The event history remains in the log. If you find a genuine new issue, **stop**; send us its section, item number, and short note. The data gate must not approve an unresolved issue.


In [ ]:
show_progress()


## 7. Final operator approval — Abhinand only, after READY

This cell **does not approve automatically**. Abhinand runs it only after personally reviewing all 85 items, accepting the six documented implicit exclusions, resolving every new issue, and matching the Drive PRD to the governance lock. It writes a private review receipt and the exact data-gate receipt checked by the inference runner. If unsure about any item, leave it unrun and share the issue.


In [ ]:
def approve_data_gate():
    assert REVIEWER == 'Abhinand', 'Only Abhinand can record the operator approval'
    assert show_progress(), 'Review is not READY; resolve pending items/issues first'
    assert json.loads(review_path.read_text()) == review_log, 'Review log changed outside this notebook; restart and inspect it'
    assert drive_prd.is_file() and sha(drive_prd) == expected_prd_sha, 'Drive PRD is stale. Complete Section 5a.'
    assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip(), 'Repository is dirty'
    assert validation['structural_checks'] == 'passed' and lengths['status'] == 'passed'
    assert lengths['validation_manifest_sha256'] == sha(validation_path)
    assert validation['review_packet_sha256'] == sha(packet_path)
    for relative, digest in lengths['inputs_sha256'].items():
        assert sha(SHARED / relative) == digest, f'Changed input: {relative}'
    # Rerun the immutable quality check immediately before approval.
    subprocess.run(['make', 'cab-quality-check', f'ROOT={SHARED}'], cwd=REPO, check=True)
    assert sha(quality_path) == review_header['quality_manifest_sha256'], 'Quality rule changed since review'
    phrase = 'I, Abhinand, personally reviewed all 85 items and accept the six documented implicit exclusions.'
    response = input('Type this exact statement to approve: ' + phrase + '\n> ').strip()
    assert response == phrase, 'No approval recorded; exact attestation was not entered'
    latest = [current(section, index) for section, count in sections.items() for index in range(count)]
    review_receipt = {'schema_version': 1, 'reviewed_by': REVIEWER,
                      'reviewed_at_utc': max(event['reviewed_at_utc'] for event in review_log['events']),
                      'review_packet_sha256': sha(packet_path),
                      'review_log_sha256': sha(review_path),
                      'cab_implicit_quality_manifest_sha256': sha(quality_path),
                      'review_counts': sections,
                      'documented_excluded_implicit_scenario_ids': sorted(quality['excluded_implicit_scenario_ids']),
                      'reviewed_development_exclusion_ids': sorted(event['id'] for event in latest if event['decision'] == 'x'),
                      'unresolved_new_issues': [],
                      'disposition': 'approved_with_documented_implicit_exclusions',
                      'review_type': 'single_operator_not_independent_replication'}
    gate = {'schema_version': 1,
            'validation_manifest_sha256': sha(validation_path),
            'token_length_manifest_sha256': sha(length_path),
            'cab_implicit_quality_manifest_sha256': sha(quality_path),
            'governance_lock_sha256': sha(lock_path),
            'token_length_status': 'passed',
            'human_review_status': 'approved',
            'approved_by': ['Abhinand']}
    review_receipt_path = SHARED / 'manifests/data_operator_review_receipt.json'
    gate_path = SHARED / 'manifests/data_gate_approval.json'
    for path, document in ((review_receipt_path, review_receipt), (gate_path, gate)):
        content = (json.dumps(document, indent=2, sort_keys=True) + '\n').encode()
        if path.exists():
            assert path.read_bytes() == content, f'Existing receipt differs: {path}; do not overwrite it'
        else:
            path.write_bytes(content)
    import importlib
    sys.path.insert(0, str(REPO / 'src'))
    from inference.runner import _check_test_gate
    _check_test_gate(SHARED)
    print('PASS: approved data gate verified by inference runner')
    print('Private review receipt:', review_receipt_path)
    print('Private gate receipt:', gate_path)

# Remove the leading # and run this cell only after Section 6 says READY.
# approve_data_gate()


## What to send back

Send only the short output from **Section 4** and the final `show_progress()` status. If you completed Section 7, tell me whether it printed `PASS: approved data gate verified by inference runner`. Do not paste private prompts, PII, canary values, or the entire review log into chat. I can then inspect the manifests in Drive and set up the development pilot and L4 model runs.
